# 실험 X: negative 범주 하나 빼고 학습 — 매핑 가설 확인 (논문에 없는 추가)
설계·예측은 `EXPERIMENT_DESIGN.md` 4.3절 (돌리기 전에 적어 둠).

**매핑 가설**: `<nexa_20>` 으로 학습한 모델은 "목록에 없음" 이라는 개념이 아니라, 학습 negative 에 나온 **종류의 요청 → `<nexa_20>`** 을 외운다.
→ 학습에서 뺀 범주는 다른 범주보다 덜 거절해야 한다. OE(노트북 10) 방식이면 그 차이가 작아야 한다.

| 모델 | negative | 학습 방식 |
|---|---|---|
| `X_token_writing` | writing 범주(250개) 뺀 1,750개 | `<nexa_20>` (ours 와 같음) |
| `X_oe_writing` | 〃 | OE (노트북 10 과 같음) |
| `X_token_smart_home` | smart_home 범주(250개) 뺀 1,750개 | `<nexa_20>` |
| `X_oe_smart_home` | 〃 | OE |
| 비교: `ours`, `OE` | 전부 2,000개 | 각 방식 |

- writing 은 함수와 먼 범주, smart_home 은 지원 함수(Nest 온도·문 잠금·Nest Hub)와 헷갈리는 near-miss 범주
- positive 2,000개·나머지 설정은 그대로. negative 가 250개 적어 step 수가 조금 줄어든다 (750 → 705)
- 채점: **우리 test(dev) 만**. 범주 라벨이 우리 test 에만 있다 (`data/neg_categories.json`, 범주마다 50개). 공식 210·MASSIVE·BFCL 은 안 함
- 먼저 노트북 10 (OE) 을 끝내 두면 `OE` 비교 칸이 채워진다. 없으면 그 칸만 빠진다

런타임: **A100 GPU + 고용량 RAM (80GB)**. 🔑 Secrets 에 `HF_TOKEN`. 학습 4번 + 생성 4번이라 1시간 넘게 걸릴 수 있다 (세션이 끊기면 아래 "이어서 하기" 참고)
Drive 용량: 모델 하나 약 5GB × 4. 모자라면 `DELETE_AFTER_EVAL = True`

## 준비 (그냥 실행)

In [ ]:
# 다른 노트북과 같은 버전 (최신 5.x 는 API 가 바뀌어 에러가 난다)
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, os, json, datetime; print("transformers", transformers.__version__)  # 4.56.2 여야 함

DRIVE = "/content/drive/MyDrive/octopus_v2"
DEV_EVAL = f"{DRIVE}/runs/eval"
DEV_TEST = "data/test.jsonl"

CATS = ["writing", "smart_home"]       # 뺄 범주 (src/neg_categories.SHORT)
LOSSES = ["token", "oe"]               # token = <nexa_20> (ours 방식), oe = 노트북 10 방식
RUNS = {f"X_{loss}_{cat}": (loss, cat) for cat in CATS for loss in LOSSES}
DELETE_AFTER_EVAL = False              # True 면 채점 끝난 모델을 Drive 에서 지운다 (probs·preds 는 남김)

def run_dir(name): return f"{DRIVE}/runs/{name}"
def done_training(name): return os.path.exists(f"{run_dir(name)}/train_args.json")
def done_eval(name): return os.path.exists(f"{DEV_EVAL}/{name}/probs.jsonl") and sum(1 for _ in open(f"{DEV_EVAL}/{name}/probs.jsonl")) == 800
print({n: ("채점 끝" if done_eval(n) else "학습 끝" if done_training(n) else "아직") for n in RUNS})
# GPU 확인: ours·A00 과 같은 조건(fp32 가중치 + AdamW, batch 16)은 A100 80GB 에서만 들어간다.
# 40GB 는 가중치 10 + 기울기 10 + Adam 상태 20 = 약 40GB 라 batch 를 줄여도 안 들어감 (8-bit Adam 등은 조건이 달라져 비교가 깨짐)
import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
EXTRA = ""
assert GPU_GB > 60, f"GPU {GPU_GB:.0f} GiB: A100 80GB 가 필요하다. 런타임 연결 해제 후 A100 + 고용량 RAM 으로 다시 연결"
print(f"GPU {GPU_GB:.0f} GiB OK")

---
## 1. 데이터 확인
**예측**: writing 을 빼면 학습 negative 는 몇 개? 우리 test 의 writing negative 는 몇 개?
> (여기에 예측)

In [ ]:
!python -m src.neg_categories show

## 2. 시험 학습 (첫 모델만, 30 step)
확인할 것: `[데이터] negative 범주 'writing' 제외: 4000 → 3750개`, oe 쪽은 `pos_loss`·`oe_loss` 가 따로 찍힘

In [ ]:
!python -m src.train --neg-loss token --exclude-neg-category writing --max-steps 30 --output-dir /content/runs/try_X {EXTRA}
!python -m src.train --neg-loss oe --exclude-neg-category writing --max-steps 30 --output-dir /content/runs/try_X {EXTRA}

---
## 3. 본 학습 → 우리 test 생성 (모델마다)
이미 끝난 모델은 건너뛴다. **세션이 끊기면** 준비 셀 두 개를 다시 실행하고 이 셀을 다시 실행하면 이어서 한다.

In [ ]:
for name, (loss, cat) in RUNS.items():
    print(f"\n==================== {name}  (학습 방식 {loss}, 뺀 범주 {cat})")
    if not done_training(name):
        print("학습 시작", datetime.datetime.now())
        !python -m src.train --neg-loss {loss} --exclude-neg-category {cat} --output-dir {run_dir(name)} {EXTRA}
        print("학습 끝", datetime.datetime.now())
    if not done_eval(name):
        out = f"{DEV_EVAL}/{name}"
        !python -m src.evaluate generate --model "{run_dir(name)}" --test-file {DEV_TEST} --out {out}/preds.jsonl
        !python -m src.evaluate score --preds {out}/preds.jsonl --test-file {DEV_TEST}
        !python -m src.probs generate --model "{run_dir(name)}" --test-file {DEV_TEST} --preds {out}/preds.jsonl --out {out}/probs.jsonl
    if DELETE_AFTER_EVAL and done_eval(name):
        !rm -rf "{run_dir(name)}"/*.safetensors

---
## 4. 범주별 거절 — 핵심 표
각 칸 = **greedy 거절 / 문턱값 거절 / 50**, 그리고 AUROC (positive 400 vs 그 범주 50). `*` = 그 모델이 학습에서 뺀 범주.
문턱값은 모델마다 우리 test positive 의 90% 가 통과하도록 (negative 는 문턱값 정하는 데 안 씀).

읽는 법
- token 모델: greedy 거절(`<nexa_20>` 출력)이 `*` 범주에서 크게 떨어지면 → **매핑** (그 범주를 거절 클래스로 외웠었다)
- oe 모델: greedy 는 원래 0. 문턱값 거절과 AUROC 가 `*` 범주에서 token 모델보다 덜 떨어지면 → OE 가 매핑을 줄였다
- writing(먼 범주) 은 A00(negative 0개)도 50/50 거절했다 → 매핑이 없어도 거절될 수 있음. smart_home 이 진짜 시험대

**예측**: smart_home 을 뺀 token 모델은 smart_home negative 50개 중 몇 개를 greedy 로 거절할까? (전체 학습 ours 는 49/50)
> (여기에 예측)

In [ ]:
def has(name): return os.path.exists(f"{DEV_EVAL}/{name}/probs.jsonl")
models = [("ours", "irr"), ("OE", "msp")] + [(n, "irr" if l == "token" else "msp") for n, (l, c) in RUNS.items()]
models = [(n, s) for n, s in models if has(n)]
print("표에 넣는 모델:", models)
args = " ".join(f"--model {n}:{s} {DEV_EVAL}/{n}/probs.jsonl" for n, s in models)
held = " ".join(f"--held-out {n}={c}" for n, (l, c) in RUNS.items())
!python -m src.negcat_report --test-file {DEV_TEST} {args} {held} --out {DEV_EVAL}/X_negcat_report.json | tee {DEV_EVAL}/X_negcat_report.txt

## 5. 전체 정확도도 확인 (범주를 빼서 다른 데가 망가지지 않았나)

In [ ]:
for name in ["ours"] + list(RUNS):
    if has(name):
        print(f"\n===== {name}")
        !python -m src.evaluate score --preds {DEV_EVAL}/{name}/preds.jsonl --test-file {DEV_TEST} | head -8

---
## 끝나면
결과: Drive `runs/eval/X_negcat_report.txt`·`.json` (범주별 표), `runs/eval/X_*/probs.jsonl`.
랩탑에서 Claude 에게 "매핑 실험 결과 분석해줘" 라고 하면 Drive 에서 받아 설계 문서 4.3절 예측과 비교한다.